In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv('language.csv')

In [3]:
df.head(5)

,Text,language
0,klement gottwaldi surnukeha palsameeriti ning ...,Estonian
1,sebes joseph pereira thomas på eng the jesuit...,Swedish
2,ถนนเจริญกรุง อักษรโรมัน thanon charoen krung เ...,Thai
3,விசாகப்பட்டினம் தமிழ்ச்சங்கத்தை இந்துப் பத்திர...,Tamil
4,de spons behoort tot het geslacht haliclona en...,Dutch


In [4]:
df.shape

(22000, 2)

In [5]:
df.isnull().sum()

,0
Text,0
language,0


In [6]:
df['language'].nunique()

22

In [7]:
df['language'].unique()

array(['Estonian', 'Swedish', 'Thai', 'Tamil', 'Dutch', 'Japanese',
       'Turkish', 'Latin', 'Urdu', 'Indonesian', 'Portugese', 'French',
       'Chinese', 'Korean', 'Hindi', 'Spanish', 'Pushto', 'Persian',
       'Romanian', 'Russian', 'English', 'Arabic'], dtype=object)

In [8]:
df['language'].value_counts()

,count
language,
Estonian,1000
Swedish,1000
English,1000
Russian,1000
Romanian,1000
Persian,1000
Pushto,1000
Spanish,1000
Hindi,1000


In [9]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import string

In [10]:
class TextCleaner(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        return [self.clean(text) for text in X]

    def clean(self, text):

        # Step 1 — Convert to string
        text = str(text)

        # Step 2 — Lowercase
        text = text.lower()

        # Step 3 — Remove URLs
        text = re.sub(r'http\S+|www\S+', '', text)

        # Step 4 — Remove emails
        text = re.sub(r'\S+@\S+', '', text)

        # Step 5 — Remove numbers
        text = re.sub(r'\d+', '', text)

        # Step 6 — Remove ONLY ASCII punctuation
        # Indian + other scripts safe rahenge
        text = re.sub(r'[!"#$%&\'()*+,\-./:;<=>?@\[\\\]^_`{|}~]', '', text)

        # Step 7 — Remove extra whitespace
        text = re.sub(r'\s+', ' ', text).strip()

        return text

In [11]:
cleaner = TextCleaner()

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

# Prepare X and y
X = df['Text']
y = df['language']

# Train Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")

Training samples : 17600
Testing samples  : 4400


In [13]:
# Pipeline 1 — Naive Bayes
nb_pipeline = Pipeline([
    ('cleaner',    TextCleaner()),
    ('vectorizer', TfidfVectorizer(
                       analyzer='char',
                       ngram_range=(2, 4),
                       max_features=50000
                   )),
    ('model',      MultinomialNB())
])

# Pipeline 2 — Logistic Regression
lr_pipeline = Pipeline([
    ('cleaner',    TextCleaner()),
    ('vectorizer', TfidfVectorizer(
                       analyzer='char',
                       ngram_range=(2, 5),
                       max_features=50000
                   )),
    ('model',      LogisticRegression(max_iter=1000, random_state=42))
])

In [14]:
import time

# Train NB
print("Training Naive Bayes...")
nb_start = time.time()
nb_pipeline.fit(X_train, y_train)
nb_time  = time.time() - nb_start
print(f"✅ NB Trained in {nb_time:.2f} sec")

print()

# Train LR
print("Training Logistic Regression...")
lr_start = time.time()
lr_pipeline.fit(X_train, y_train)
lr_time  = time.time() - lr_start
print(f"✅ LR Trained in {lr_time:.2f} sec")

Training Naive Bayes...
✅ NB Trained in 71.44 sec

Training Logistic Regression...
✅ LR Trained in 81.79 sec


In [15]:
# Predictions
nb_pred  = nb_pipeline.predict(X_test)
lr_pred  = lr_pipeline.predict(X_test)

In [16]:
# Accuracy
nb_accuracy = accuracy_score(y_test, nb_pred)
lr_accuracy = accuracy_score(y_test, lr_pred)

In [17]:
lr_accuracy

0.9825

In [18]:
nb_accuracy

0.9756818181818182

In [19]:
import joblib

joblib.dump(nb_pipeline, 'NB_language_pipeline.pkl')
print("✅ NB Pipeline saved!")

✅ NB Pipeline saved!


In [20]:
def language_probability(text):

    # ── Step 1: Get the detected language ──────────────────
    detected  = nb_pipeline.predict([text])[0]

    # ── Step 2: Get probability scores for all languages ───
    probs     = nb_pipeline.predict_proba([text])[0]

    # ── Step 3: Get all language names ─────────────────────
    languages = nb_pipeline.classes_

    # ── Step 4: Pair each language with its probability ────
    lang_prob = sorted(
                    zip(languages, probs),
                    key    = lambda x: x[1],
                    reverse= True
                )

    # ── Step 5: Check how confident the model is ───────────
    top_score = lang_prob[0][1]

    if top_score >= 0.75:
        status = "✅ High Confidence"
    elif top_score >= 0.50:
        status = "⚠️  Medium Confidence"
    else:
        status = "❌ Low Confidence"

    # ── Step 6: Display the results ────────────────────────
    print("=" * 50)
    print(f"  Input Text   : {text}")
    print(f"  Detected     : {detected}")
    print(f"  Status       : {status}")
    print("\n  Top 5 Probabilities:")
    print("  " + "-" * 45)

    for lang, prob in lang_prob[:5]:
        bar = "█" * int(prob * 40)
        gap = "░" * (40 - int(prob * 40))
        print(f"  {lang:<18} {bar}{gap}  {prob*100:.1f}%")

    print("=" * 50)

    return detected

# ── Take input and call function ────────────────────
text = input("Enter text : ")
language_probability(text)        # ← this was missing

Enter text : আমি তোমাকে ভালোবাসি এবং তোমাকে বিয়ে করতে চাই। তুমি কি আমার হবে? আমি তোমাকে সবসময় সুরক্ষিত রাখব।
  Input Text   : আমি তোমাকে ভালোবাসি এবং তোমাকে বিয়ে করতে চাই। তুমি কি আমার হবে? আমি তোমাকে সবসময় সুরক্ষিত রাখব।
  Detected     : Hindi
  Status       : ⚠️  Medium Confidence

  Top 5 Probabilities:
  ---------------------------------------------
  Hindi              ████████████████████████░░░░░░░░░░░░░░░░  60.9%
  Chinese            ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  2.2%
  Japanese           ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  2.1%
  Korean             ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  2.1%
  Thai               ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  1.9%


'Hindi'

In [21]:
!pip install "numpy<2.0"
!pip install fasttext
!pip install transformers sentencepiece


In [22]:
import fasttext
import urllib.request
import os

In [23]:
model_path = 'lid.176.ftz'

if not os.path.exists(model_path):
    print("Downloading fastText model...")
    urllib.request.urlretrieve(
        'https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz',
        model_path
    )
    print("✅ Downloaded!")
else:
    print("✅ Already exists!")

fasttext.FastText.eprint = lambda x: None
ft_model = fasttext.load_model(model_path)
print("✅ FastText loaded!")

✅ Downloaded!
✅ FastText loaded!


In [24]:
language_codes = {
    'Arabic'    : 'ar',
    'Danish'    : 'da',
    'Dutch'     : 'nl',
    'English'   : 'en',
    'French'    : 'fr',
    'German'    : 'de',
    'Greek'     : 'el',
    'Hindi'     : 'hi',
    'Italian'   : 'it',
    'Kannada'   : 'kn',
    'Malayalam' : 'ml',
    'Portugese' : 'pt',
    'Russian'   : 'ru',
    'Spanish'   : 'es',
    'Swahili'   : 'sw',
    'Swedish'   : 'sv',
    'Tamil'     : 'ta',
    'Telugu'    : 'te',
    'Thai'      : 'th',
    'Turkish'   : 'tr',
    'Urdu'      : 'ur',
    'Romanian'  : 'ro',
    'Serbian'   : 'sr',
    'Macedonian': 'mk',
    'Bulgarian' : 'bg',
    'Ukrainian' : 'uk',
    'Polish'    : 'pl',
    'Czech'     : 'cs',
    'Croatian'  : 'hr',
    'Hungarian' : 'hu',
    'Finnish'   : 'fi',
    'Norwegian' : 'no',
    'Catalan'   : 'ca',
    'Chinese'   : 'zh',
    'Japanese'  : 'ja',
    'Korean'    : 'ko',
    'Indonesian': 'id',
    'Hebrew'    : 'he',
    'Persian'   : 'fa',
    'Bengali'   : 'bn',
    'Vietnamese': 'vi',
}

code_to_language = {v: k for k, v in language_codes.items()}
print(f"✅ Total languages mapped: {len(language_codes)}")

✅ Total languages mapped: 41


In [25]:
from transformers import MarianMTModel, MarianTokenizer

def nlp_translate(text, source_code, target_code):

    # Same language check
    if source_code == target_code:
        return text

    # Level 1 — Direct model
    try:
        model_name = f'Helsinki-NLP/opus-mt-{source_code}-{target_code}'
        tokenizer  = MarianTokenizer.from_pretrained(model_name)
        model      = MarianMTModel.from_pretrained(model_name)
        tokens     = tokenizer(
                         [text],
                         return_tensors = 'pt',
                         padding        = True,
                         truncation     = True
                     )
        out        = model.generate(**tokens)
        return tokenizer.decode(out[0], skip_special_tokens=True)
    except:
        print(f"  ⚠️  Direct model not found")

    # Level 2 — tc-big model
    try:
        model_name = f'Helsinki-NLP/opus-mt-tc-big-{source_code}-{target_code}'
        tokenizer  = MarianTokenizer.from_pretrained(model_name)
        model      = MarianMTModel.from_pretrained(model_name)
        tokens     = tokenizer(
                         [text],
                         return_tensors = 'pt',
                         padding        = True,
                         truncation     = True
                     )
        out        = model.generate(**tokens)
        return tokenizer.decode(out[0], skip_special_tokens=True)
    except:
        print(f"  ⚠️  tc-big model not found")

    # Level 3 — Via English
    try:
        print(f"  Trying via English...")
        t1       = MarianTokenizer.from_pretrained(
                       f'Helsinki-NLP/opus-mt-{source_code}-en')
        m1       = MarianMTModel.from_pretrained(
                       f'Helsinki-NLP/opus-mt-{source_code}-en')
        tok1     = t1(
                       [text],
                       return_tensors = 'pt',
                       padding        = True,
                       truncation     = True
                   )
        eng_text = t1.decode(
                       m1.generate(**tok1)[0],
                       skip_special_tokens=True
                   )

        if target_code == 'en':
            return eng_text

        t2   = MarianTokenizer.from_pretrained(
                   f'Helsinki-NLP/opus-mt-en-{target_code}')
        m2   = MarianMTModel.from_pretrained(
                   f'Helsinki-NLP/opus-mt-en-{target_code}')
        tok2 = t2(
                   [eng_text],
                   return_tensors = 'pt',
                   padding        = True,
                   truncation     = True
               )
        return t2.decode(
                   m2.generate(**tok2)[0],
                   skip_special_tokens=True
               )
    except:
        print(f"  ⚠️  Via English also failed")

    # Level 4 — Romance group
    try:
        print(f"  Trying Romance group...")
        t3   = MarianTokenizer.from_pretrained(
                   'Helsinki-NLP/opus-mt-ROMANCE-en')
        m3   = MarianMTModel.from_pretrained(
                   'Helsinki-NLP/opus-mt-ROMANCE-en')
        tok3 = t3(
                   [f">>{source_code}<< {text}"],
                   return_tensors = 'pt',
                   padding        = True,
                   truncation     = True
               )
        return t3.decode(
                   m3.generate(**tok3)[0],
                   skip_special_tokens=True
               )
    except:
        pass

    return "❌ Translation not available for this language pair"

In [26]:
def smart_detect(text):

    # Step 1 — Check length
    if len(text.strip()) < 3:
        print("⚠️  Text too short")
        return None, None

    # Step 2 — NB prediction
    detected  = nb_pipeline.predict([text])[0]
    probs     = nb_pipeline.predict_proba([text])[0]
    languages = nb_pipeline.classes_

    lang_prob = sorted(
                    zip(languages, probs),
                    key     = lambda x: x[1],
                    reverse = True
                )
    top_score = lang_prob[0][1]

    # Step 3 — FastText cross verify
    try:
        clean_text    = text.replace('\n', ' ').strip()
        predictions   = ft_model.predict(clean_text, k=3)
        ft_lang_code  = predictions[0][0].replace('__label__', '')
        ft_confidence = float(predictions[1][0])
        ft_lang_name  = code_to_language.get(ft_lang_code)
        nb_ft_match   = (ft_lang_name == detected)

        print(f"\n  FastText Top 3:")
        print("  " + "-" * 40)
        for code, score in zip(predictions[0], predictions[1]):
            c    = code.replace('__label__', '')
            name = code_to_language.get(c, c)
            print(f"  {name:<20} ({c})  {float(score)*100:.1f}%")

    except Exception as e:
        ft_lang_code  = None
        ft_lang_name  = None
        ft_confidence = 0.0
        nb_ft_match   = False
        print(f"  FastText Error : {e}")

    # Step 4 — Dono low confidence
    if ft_confidence < 0.50 and top_score < 0.60:
        print(f"\n  ⚠️  Both models low confidence")
        print(f"  Please enter longer text")
        return None, None

    # Step 5 — NB + FastText match
    if top_score >= 0.60 and nb_ft_match:
        print("\n" + "=" * 50)
        print(f"  Input Text    : {text}")
        print(f"  Detected      : {detected}")
        print(f"  Method        : ✅ NB + FastText Verified")
        print(f"\n  Top 5 Probabilities:")
        print("  " + "-" * 45)
        for lang, prob in lang_prob[:5]:
            bar = "█" * int(prob * 40)
            gap = "░" * (40 - int(prob * 40))
            print(f"  {lang:<20} {bar}{gap}  {prob*100:.1f}%")
        print("=" * 50)
        return detected, language_codes.get(detected)

    # Step 6 — NB confident only
    elif top_score >= 0.60 and not nb_ft_match:
        print("\n" + "=" * 50)
        print(f"  Input Text    : {text}")
        print(f"  Detected      : {detected}")
        print(f"  Method        : ✅ NB Model (High Confidence)")
        print(f"\n  Top 5 Probabilities:")
        print("  " + "-" * 45)
        for lang, prob in lang_prob[:5]:
            bar = "█" * int(prob * 40)
            gap = "░" * (40 - int(prob * 40))
            print(f"  {lang:<20} {bar}{gap}  {prob*100:.1f}%")
        print("=" * 50)
        return detected, language_codes.get(detected)

    # Step 7 — FastText use karo
    else:
        print(f"\n  NB Result  : {detected} ({top_score*100:.1f}%)")
        print(f"  FastText   : {ft_lang_name} ({ft_confidence*100:.1f}%)")
        print(f"  Method     : ⚠️  FastText used")
        if ft_lang_code:
            language_name = code_to_language.get(
                                ft_lang_code,
                                f"Lang_{ft_lang_code.upper()}"
                            )
            return language_name, ft_lang_code
        return None, None

In [27]:
def detect_and_translate(text, target_lang):

    # Case fix
    target_lang = target_lang.strip().title()

    # Step 1 — Detect
    detected, source_code = smart_detect(text)
    if not detected:
        return

    # Step 2 — Target code
    target_code = language_codes.get(target_lang)
    if not target_code:
        print(f"❌ '{target_lang}' not supported")
        return

    # Step 3 — Same language check
    if detected == target_lang:
        print(f"ℹ️  Already in {target_lang}")
        return

    # Step 4 — Source code fix
    if not source_code:
        try:
            pred        = ft_model.predict(
                              text.replace('\n',' ').strip(), k=1)
            source_code = pred[0][0].replace('__label__', '')
        except:
            print("❌ Source code not found")
            return

    # Step 5 — Translate
    print(f"\n  Translating...")
    translated = nlp_translate(text, source_code, target_code)

    # Step 6 — Output
    print("\n" + "=" * 50)
    print(f"  Original Text : {text}")
    print(f"  Detected Lang : {detected}")
    print(f"  Target Lang   : {target_lang}")
    print(f"  Translated    : {translated}")
    print("=" * 50)

In [31]:
text        = input("Enter text        : ")
target_lang = input("Enter target lang : ")

detect_and_translate(text, target_lang)

Enter text        : I love you and I want to marry you. Will you be mine? I will always keep you safe
Enter target lang : Hindi

  FastText Top 3:
  ----------------------------------------
  English              (en)  99.3%
  Portugese            (pt)  0.1%
  Spanish              (es)  0.1%

  Input Text    : I love you and I want to marry you. Will you be mine? I will always keep you safe
  Detected      : English
  Method        : ✅ NB + FastText Verified

  Top 5 Probabilities:
  ---------------------------------------------
  English              █████████████████████████████████████░░░  93.1%
  Turkish              ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  1.5%
  Dutch                ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  0.9%
  Portugese            ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  0.9%
  Indonesian           ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  0.9%

  Translating...


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  306MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]


  Original Text : I love you and I want to marry you. Will you be mine? I will always keep you safe
  Detected Lang : English
  Target Lang   : Hindi
  Translated    : मैं तुझ से प्रेम रखता हूं, और मैं तुझ से विवाह करना चाहता हूं; क्या तू मेरा रहेगा? मैं सदा तुझे सुरक्षित रखूंगा।


In [29]:
#For Deployment

In [35]:
import streamlit as st
import joblib
import fasttext
import urllib.request
import os
import re
import string
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from transformers import MarianMTModel, MarianTokenizer

# ─────────────────────────────────────────────
# PAGE CONFIG
# ─────────────────────────────────────────────
st.set_page_config(
    page_title="LinguaDetect AI",
    page_icon="🌐",
    layout="wide",
    initial_sidebar_state="collapsed"
)

# ─────────────────────────────────────────────
# CUSTOM CSS
# ─────────────────────────────────────────────
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Syne:wght@400;600;700;800&family=DM+Sans:wght@300;400;500&display=swap');

/* ── Root Variables ── */
:root {
    --bg:        #0a0a0f;
    --surface:   #12121a;
    --card:      #1a1a26;
    --border:    #2a2a3d;
    --accent:    #6c63ff;
    --accent2:   #ff6584;
    --accent3:   #43e97b;
    --text:      #e8e8f0;
    --muted:     #6b6b8a;
}

/* ── Global ── */
html, body, [class*="css"] {
    font-family: 'DM Sans', sans-serif;
    background:  var(--bg) !important;
    color:       var(--text) !important;
}

/* ── Hide Streamlit Defaults ── */
#MainMenu, footer, header { visibility: hidden; }
.block-container { padding: 2rem 3rem !important; max-width: 1200px; }

/* ── Hero Section ── */
.hero {
    text-align: center;
    padding: 3rem 0 2rem;
    position: relative;
}
.hero-badge {
    display: inline-block;
    background: linear-gradient(135deg, #6c63ff22, #ff658422);
    border: 1px solid #6c63ff44;
    border-radius: 100px;
    padding: 6px 18px;
    font-size: 0.75rem;
    letter-spacing: 2px;
    text-transform: uppercase;
    color: var(--accent);
    margin-bottom: 1.5rem;
}
.hero-title {
    font-family: 'Syne', sans-serif;
    font-size: clamp(2.5rem, 5vw, 4rem);
    font-weight: 800;
    background: linear-gradient(135deg, #fff 0%, #6c63ff 50%, #ff6584 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    line-height: 1.1;
    margin-bottom: 1rem;
}
.hero-sub {
    color: var(--muted);
    font-size: 1.1rem;
    font-weight: 300;
    max-width: 500px;
    margin: 0 auto 2rem;
}

/* ── Stats Bar ── */
.stats-bar {
    display: flex;
    justify-content: center;
    gap: 3rem;
    margin: 1.5rem 0 2.5rem;
    flex-wrap: wrap;
}
.stat-item {
    text-align: center;
}
.stat-number {
    font-family: 'Syne', sans-serif;
    font-size: 1.8rem;
    font-weight: 700;
    background: linear-gradient(135deg, #6c63ff, #ff6584);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}
.stat-label {
    font-size: 0.75rem;
    color: var(--muted);
    text-transform: uppercase;
    letter-spacing: 1px;
}

/* ── Cards ── */
.glass-card {
    background:    var(--card);
    border:        1px solid var(--border);
    border-radius: 16px;
    padding:       1.5rem;
    margin-bottom: 1rem;
}
.section-label {
    font-size:      0.7rem;
    letter-spacing: 2px;
    text-transform: uppercase;
    color:          var(--muted);
    margin-bottom:  0.5rem;
}

/* ── Input Area ── */
.stTextArea textarea {
    background:    var(--surface) !important;
    border:        1px solid var(--border) !important;
    border-radius: 12px !important;
    color:         var(--text) !important;
    font-family:   'DM Sans', sans-serif !important;
    font-size:     1rem !important;
    padding:       1rem !important;
    resize:        vertical !important;
    transition:    border-color 0.3s !important;
}
.stTextArea textarea:focus {
    border-color: var(--accent) !important;
    box-shadow: 0 0 0 3px #6c63ff22 !important;
}

/* ── Select Box ── */
.stSelectbox > div > div {
    background:    var(--surface) !important;
    border:        1px solid var(--border) !important;
    border-radius: 12px !important;
    color:         var(--text) !important;
}

/* ── Button ── */
.stButton > button {
    background:    linear-gradient(135deg, #6c63ff, #9c88ff) !important;
    color:         white !important;
    border:        none !important;
    border-radius: 12px !important;
    padding:       0.75rem 2rem !important;
    font-family:   'Syne', sans-serif !important;
    font-weight:   600 !important;
    font-size:     1rem !important;
    letter-spacing: 0.5px !important;
    width:         100% !important;
    transition:    all 0.3s !important;
    cursor:        pointer !important;
}
.stButton > button:hover {
    transform:  translateY(-2px) !important;
    box-shadow: 0 8px 25px #6c63ff44 !important;
}

/* ── Result Cards ── */
.result-detected {
    background: linear-gradient(135deg, #6c63ff11, #9c88ff11);
    border: 1px solid #6c63ff44;
    border-radius: 12px;
    padding: 1.2rem 1.5rem;
    margin-bottom: 1rem;
}
.result-translated {
    background: linear-gradient(135deg, #43e97b11, #38f9d711);
    border: 1px solid #43e97b44;
    border-radius: 12px;
    padding: 1.2rem 1.5rem;
    margin-bottom: 1rem;
}
.result-label {
    font-size: 0.7rem;
    letter-spacing: 2px;
    text-transform: uppercase;
    margin-bottom: 0.4rem;
}
.result-detected .result-label { color: #9c88ff; }
.result-translated .result-label { color: #43e97b; }
.result-value {
    font-family: 'Syne', sans-serif;
    font-size: 1.3rem;
    font-weight: 700;
    color: var(--text);
}

/* ── Probability Bars ── */
.prob-container {
    margin-top: 0.5rem;
}
.prob-row {
    display: flex;
    align-items: center;
    gap: 0.8rem;
    margin-bottom: 0.6rem;
}
.prob-lang {
    width: 100px;
    font-size: 0.85rem;
    color: var(--text);
    text-align: right;
}
.prob-bar-bg {
    flex: 1;
    height: 8px;
    background: var(--surface);
    border-radius: 100px;
    overflow: hidden;
}
.prob-bar-fill {
    height: 100%;
    border-radius: 100px;
    background: linear-gradient(90deg, #6c63ff, #9c88ff);
    transition: width 0.8s ease;
}
.prob-pct {
    width: 45px;
    font-size: 0.8rem;
    color: var(--muted);
    text-align: right;
}

/* ── Method Badge ── */
.method-badge {
    display: inline-flex;
    align-items: center;
    gap: 6px;
    background: #43e97b22;
    border: 1px solid #43e97b44;
    border-radius: 100px;
    padding: 4px 12px;
    font-size: 0.75rem;
    color: #43e97b;
    margin-bottom: 1rem;
}
.method-badge-warn {
    background: #ffb74d22;
    border-color: #ffb74d44;
    color: #ffb74d;
}

/* ── Info Cards ── */
.info-grid {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 1rem;
    margin-top: 2rem;
}
.info-card {
    background: var(--card);
    border: 1px solid var(--border);
    border-radius: 12px;
    padding: 1.2rem;
    text-align: center;
}
.info-icon {
    font-size: 1.8rem;
    margin-bottom: 0.5rem;
}
.info-title {
    font-family: 'Syne', sans-serif;
    font-size: 0.9rem;
    font-weight: 600;
    margin-bottom: 0.3rem;
}
.info-desc {
    font-size: 0.78rem;
    color: var(--muted);
    line-height: 1.5;
}

/* ── Divider ── */
.divider {
    border: none;
    border-top: 1px solid var(--border);
    margin: 2rem 0;
}

/* ── Spinner ── */
.stSpinner > div {
    border-top-color: var(--accent) !important;
}

/* ── Copy Button ── */
.copy-btn {
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: 8px;
    padding: 4px 12px;
    font-size: 0.75rem;
    color: var(--muted);
    cursor: pointer;
    float: right;
}
</style>
""", unsafe_allow_html=True)


# ─────────────────────────────────────────────
# TEXT CLEANER CLASS
# ─────────────────────────────────────────────
class TextCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        return [self.clean(text) for text in X]
    def clean(self, text):
        text = str(text)
        text = text.lower()
        text = re.sub(r'http\S+|www\S+', '', text)
        text = re.sub(r'\S+@\S+', '', text)
        text = re.sub(r'\d+', '', text)
        text = re.sub(r'[!"#$%&\'()*+,\-./:;<=>?@\[\\\]^_`{|}~]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text


# ─────────────────────────────────────────────
# LANGUAGE CODES
# ─────────────────────────────────────────────
LANGUAGE_CODES = {
    'Arabic': 'ar', 'Danish': 'da', 'Dutch': 'nl',
    'English': 'en', 'French': 'fr', 'German': 'de',
    'Greek': 'el', 'Hindi': 'hi', 'Italian': 'it',
    'Kannada': 'kn', 'Malayalam': 'ml', 'Portugese': 'pt',
    'Russian': 'ru', 'Spanish': 'es', 'Swahili': 'sw',
    'Swedish': 'sv', 'Tamil': 'ta', 'Telugu': 'te',
    'Thai': 'th', 'Turkish': 'tr', 'Urdu': 'ur',
    'Romanian': 'ro', 'Serbian': 'sr', 'Macedonian': 'mk',
    'Bulgarian': 'bg', 'Ukrainian': 'uk', 'Polish': 'pl',
    'Czech': 'cs', 'Croatian': 'hr', 'Hungarian': 'hu',
    'Finnish': 'fi', 'Norwegian': 'no', 'Catalan': 'ca',
    'Chinese': 'zh', 'Japanese': 'ja', 'Korean': 'ko',
    'Indonesian': 'id', 'Hebrew': 'he', 'Persian': 'fa',
    'Bengali': 'bn', 'Vietnamese': 'vi',
}
CODE_TO_LANG = {v: k for k, v in LANGUAGE_CODES.items()}


# ─────────────────────────────────────────────
# LOAD MODELS (Cached)
# ─────────────────────────────────────────────
@st.cache_resource
def load_nb_model():
    return joblib.load('NB_language_pipeline.pkl')

@st.cache_resource
def load_fasttext_model():
    model_path = 'lid.176.ftz'
    if not os.path.exists(model_path):
        urllib.request.urlretrieve(
            'https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz',
            model_path
        )
    fasttext.FastText.eprint = lambda x: None
    return fasttext.load_model(model_path)

@st.cache_resource
def load_translation_model(source_code, target_code):
    model_name = f'Helsinki-NLP/opus-mt-{source_code}-{target_code}'
    tokenizer  = MarianTokenizer.from_pretrained(model_name)
    model      = MarianMTModel.from_pretrained(model_name)
    return tokenizer, model


# ─────────────────────────────────────────────
# DETECTION FUNCTION
# ─────────────────────────────────────────────
def detect_language(text, nb_pipeline, ft_model):
    detected  = nb_pipeline.predict([text])[0]
    probs     = nb_pipeline.predict_proba([text])[0]
    languages = nb_pipeline.classes_

    lang_prob = sorted(
        zip(languages, probs),
        key=lambda x: x[1],
        reverse=True
    )
    top_score = lang_prob[0][1]

    try:
        clean_text    = text.replace('\n', ' ').strip()
        predictions   = ft_model.predict(clean_text, k=3)
        ft_lang_code  = predictions[0][0].replace('__label__', '')
        ft_confidence = float(predictions[1][0])
        ft_lang_name  = CODE_TO_LANG.get(ft_lang_code)
        nb_ft_match   = (ft_lang_name == detected)
    except:
        ft_lang_code  = None
        ft_lang_name  = None
        ft_confidence = 0.0
        nb_ft_match   = False

    if ft_confidence < 0.50 and top_score < 0.60:
        return None, None, lang_prob, "low"

    if top_score >= 0.60 and nb_ft_match:
        method = "verified"
    elif top_score >= 0.60:
        method = "nb_only"
    else:
        method = "fasttext"
        if ft_lang_code:
            detected = CODE_TO_LANG.get(ft_lang_code,
                           f"Lang_{ft_lang_code.upper()}")

    source_code = LANGUAGE_CODES.get(detected, ft_lang_code)
    return detected, source_code, lang_prob, method


# ─────────────────────────────────────────────
# TRANSLATION FUNCTION
# ─────────────────────────────────────────────
def translate(text, source_code, target_code):
    if source_code == target_code:
        return text

    levels = [
        f'Helsinki-NLP/opus-mt-{source_code}-{target_code}',
        f'Helsinki-NLP/opus-mt-tc-big-{source_code}-{target_code}',
    ]

    for model_name in levels:
        try:
            tok = MarianTokenizer.from_pretrained(model_name)
            mdl = MarianMTModel.from_pretrained(model_name)
            t   = tok([text], return_tensors='pt',
                      padding=True, truncation=True)
            out = mdl.generate(**t)
            return tok.decode(out[0], skip_special_tokens=True)
        except:
            continue

    # Via English
    try:
        t1   = MarianTokenizer.from_pretrained(
                   f'Helsinki-NLP/opus-mt-{source_code}-en')
        m1   = MarianMTModel.from_pretrained(
                   f'Helsinki-NLP/opus-mt-{source_code}-en')
        tok1 = t1([text], return_tensors='pt',
                  padding=True, truncation=True)
        eng  = t1.decode(m1.generate(**tok1)[0],
                         skip_special_tokens=True)

        if target_code == 'en':
            return eng

        t2   = MarianTokenizer.from_pretrained(
                   f'Helsinki-NLP/opus-mt-en-{target_code}')
        m2   = MarianMTModel.from_pretrained(
                   f'Helsinki-NLP/opus-mt-en-{target_code}')
        tok2 = t2([eng], return_tensors='pt',
                  padding=True, truncation=True)
        return t2.decode(m2.generate(**tok2)[0],
                         skip_special_tokens=True)
    except:
        pass

    return "❌ Translation not available for this language pair"


# ─────────────────────────────────────────────
# HERO SECTION
# ─────────────────────────────────────────────
st.markdown("""
<div class="hero">
    <div class="hero-badge">🤖 ML + NLP Project</div>
    <div class="hero-title">LinguaDetect AI</div>
    <div class="hero-sub">
        Detect any language instantly and translate it
        using state-of-the-art NLP models
    </div>
</div>
""", unsafe_allow_html=True)

st.markdown("""
<div class="stats-bar">
    <div class="stat-item">
        <div class="stat-number">22</div>
        <div class="stat-label">Trained Languages</div>
    </div>
    <div class="stat-item">
        <div class="stat-number">170+</div>
        <div class="stat-label">Detected via FastText</div>
    </div>
    <div class="stat-item">
        <div class="stat-number">97.5%</div>
        <div class="stat-label">Model Accuracy</div>
    </div>
    <div class="stat-item">
        <div class="stat-number">3</div>
        <div class="stat-label">ML Components</div>
    </div>
</div>
""", unsafe_allow_html=True)

st.markdown('<hr class="divider">', unsafe_allow_html=True)


# ─────────────────────────────────────────────
# LOAD MODELS
# ─────────────────────────────────────────────
with st.spinner("Loading ML models..."):
    try:
        nb_pipeline = load_nb_model()
        ft_model    = load_fasttext_model()
        models_ok   = True
    except Exception as e:
        st.error(f"❌ Model load failed: {e}")
        st.info("Make sure NB_language_pipeline.pkl is in the same folder")
        models_ok = False


# ─────────────────────────────────────────────
# MAIN LAYOUT
# ─────────────────────────────────────────────
col1, col2 = st.columns([1.1, 0.9], gap="large")

with col1:
    st.markdown('<div class="glass-card">', unsafe_allow_html=True)
    st.markdown('<div class="section-label">✍️ Input Text</div>',
                unsafe_allow_html=True)

    text_input = st.text_area(
        label     = "",
        height    = 180,
        placeholder = "Type or paste text in any language...\n\nExample: Bonjour, comment allez-vous?\nOr: नमस्ते, आप कैसे हैं?",
        label_visibility = "collapsed"
    )

    st.markdown('<div class="section-label" style="margin-top:1rem">🌍 Translate To</div>',
                unsafe_allow_html=True)

    target_lang = st.selectbox(
        label            = "",
        options          = sorted(LANGUAGE_CODES.keys()),
        index            = list(sorted(LANGUAGE_CODES.keys())).index('English'),
        label_visibility = "collapsed"
    )

    st.markdown("<br>", unsafe_allow_html=True)
    translate_btn = st.button("🚀 Detect & Translate", use_container_width=True)
    st.markdown('</div>', unsafe_allow_html=True)

    # ── Example Inputs ──
    st.markdown('<div class="section-label" style="margin-top:1rem">⚡ Try Examples</div>',
                unsafe_allow_html=True)

    examples = {
        "🇪🇸 Spanish" : "Hola, ¿cómo estás? Me llamo Juan y soy de España.",
        "🇫🇷 French"  : "Bonjour, comment allez-vous? Je suis étudiant.",
        "🇩🇪 German"  : "Guten Morgen! Wie geht es Ihnen heute?",
        "🇯🇵 Japanese": "こんにちは、お元気ですか？私は学生です。",
        "🇮🇳 Hindi"   : "नमस्ते! आप कैसे हैं? मैं ठीक हूं।",
    }

    ex_cols = st.columns(len(examples))
    for i, (lang, example) in enumerate(examples.items()):
        with ex_cols[i]:
            if st.button(lang, key=f"ex_{i}", use_container_width=True):
                st.session_state['example_text'] = example
                st.rerun()

    if 'example_text' in st.session_state:
        st.info(f"📋 Example loaded: *{st.session_state['example_text'][:50]}...*")


with col2:
    st.markdown('<div class="glass-card" style="min-height: 400px;">', unsafe_allow_html=True)
    st.markdown('<div class="section-label">📊 Results</div>', unsafe_allow_html=True)

    # Use example text if selected
    active_text = st.session_state.get('example_text', text_input)

    if translate_btn or 'example_text' in st.session_state:

        if not active_text or len(active_text.strip()) < 3:
            st.warning("⚠️ Please enter at least 3 characters")

        elif not models_ok:
            st.error("❌ Models not loaded")

        else:
            with st.spinner("Detecting language..."):
                detected, source_code, lang_prob, method = detect_language(
                    active_text, nb_pipeline, ft_model
                )

            if detected is None:
                st.warning("⚠️ Could not detect language. Please enter longer text.")

            else:
                # Method badge
                if method == "verified":
                    st.markdown(
                        '<div class="method-badge">✅ NB + FastText Verified</div>',
                        unsafe_allow_html=True
                    )
                elif method == "nb_only":
                    st.markdown(
                        '<div class="method-badge">✅ NB Model (High Confidence)</div>',
                        unsafe_allow_html=True
                    )
                else:
                    st.markdown(
                        '<div class="method-badge method-badge-warn">⚠️ FastText Backup Used</div>',
                        unsafe_allow_html=True
                    )

                # Detected Language
                st.markdown(f"""
                <div class="result-detected">
                    <div class="result-label">Detected Language</div>
                    <div class="result-value">🌐 {detected}</div>
                </div>
                """, unsafe_allow_html=True)

                # Probability Bars
                st.markdown('<div class="section-label">📈 Language Probabilities (Top 5)</div>',
                            unsafe_allow_html=True)

                prob_html = '<div class="prob-container">'
                for lang, prob in lang_prob[:5]:
                    width = int(prob * 100)
                    prob_html += f"""
                    <div class="prob-row">
                        <div class="prob-lang">{lang}</div>
                        <div class="prob-bar-bg">
                            <div class="prob-bar-fill" style="width:{width}%"></div>
                        </div>
                        <div class="prob-pct">{prob*100:.1f}%</div>
                    </div>"""
                prob_html += '</div>'
                st.markdown(prob_html, unsafe_allow_html=True)

                st.markdown("<br>", unsafe_allow_html=True)

                # Translation
                if detected.lower() == target_lang.lower():
                    st.info(f"ℹ️ Text is already in {target_lang}")
                elif not source_code:
                    st.warning("⚠️ Translation not available for this language")
                else:
                    with st.spinner("Translating..."):
                        target_code = LANGUAGE_CODES.get(target_lang)
                        translated  = translate(
                            active_text, source_code, target_code
                        )

                    st.markdown(f"""
                    <div class="result-translated">
                        <div class="result-label">Translated to {target_lang}</div>
                        <div class="result-value">{translated}</div>
                    </div>
                    """, unsafe_allow_html=True)

                # Clear example after showing
                if 'example_text' in st.session_state:
                    del st.session_state['example_text']

    else:
        st.markdown("""
        <div style="text-align:center; padding: 4rem 1rem; color: #6b6b8a;">
            <div style="font-size: 3rem; margin-bottom: 1rem">🌐</div>
            <div style="font-family: 'Syne', sans-serif; font-size: 1.1rem; margin-bottom: 0.5rem">
                Ready to Detect
            </div>
            <div style="font-size: 0.85rem; line-height: 1.6">
                Enter text on the left and click<br>
                <strong style="color:#9c88ff">Detect & Translate</strong>
            </div>
        </div>
        """, unsafe_allow_html=True)

    st.markdown('</div>', unsafe_allow_html=True)


# ─────────────────────────────────────────────
# HOW IT WORKS SECTION
# ─────────────────────────────────────────────
st.markdown('<hr class="divider">', unsafe_allow_html=True)
st.markdown("""
<div style="text-align:center; margin-bottom: 1.5rem">
    <div class="hero-badge">🔬 Under The Hood</div>
    <div style="font-family: 'Syne', sans-serif; font-size: 1.8rem; font-weight: 700; margin-top: 0.5rem">
        How It Works
    </div>
</div>

<div class="info-grid">
    <div class="info-card">
        <div class="info-icon">🧹</div>
        <div class="info-title">Text Preprocessing</div>
        <div class="info-desc">
            Custom TextCleaner pipeline removes noise,
            normalizes text and prepares it for
            ML processing
        </div>
    </div>
    <div class="info-card">
        <div class="info-icon">🤖</div>
        <div class="info-title">Naive Bayes + FastText</div>
        <div class="info-desc">
            NB model trained on 22 languages with
            97.5% accuracy. FastText handles 170+
            languages as intelligent backup
        </div>
    </div>
    <div class="info-card">
        <div class="info-icon">🌍</div>
        <div class="info-title">Helsinki NLP Translation</div>
        <div class="info-desc">
            MarianMT pretrained transformer models
            translate between language pairs with
            3-level fallback system
        </div>
    </div>
</div>
""", unsafe_allow_html=True)


# ─────────────────────────────────────────────
# PIPELINE FLOW
# ─────────────────────────────────────────────
st.markdown("<br>", unsafe_allow_html=True)
st.markdown("""
<div class="glass-card">
    <div class="section-label">⚙️ ML Pipeline Flow</div>
    <div style="display:flex; align-items:center; gap:0.5rem; flex-wrap:wrap; margin-top:0.8rem">
        <div style="background:#6c63ff22; border:1px solid #6c63ff44; border-radius:8px; padding:6px 14px; font-size:0.85rem">
            📝 Raw Text
        </div>
        <div style="color:#6b6b8a">→</div>
        <div style="background:#6c63ff22; border:1px solid #6c63ff44; border-radius:8px; padding:6px 14px; font-size:0.85rem">
            🧹 TextCleaner
        </div>
        <div style="color:#6b6b8a">→</div>
        <div style="background:#6c63ff22; border:1px solid #6c63ff44; border-radius:8px; padding:6px 14px; font-size:0.85rem">
            📊 TF-IDF Vectorizer
        </div>
        <div style="color:#6b6b8a">→</div>
        <div style="background:#6c63ff22; border:1px solid #6c63ff44; border-radius:8px; padding:6px 14px; font-size:0.85rem">
            🤖 Naive Bayes
        </div>
        <div style="color:#6b6b8a">→</div>
        <div style="background:#ff658422; border:1px solid #ff658444; border-radius:8px; padding:6px 14px; font-size:0.85rem">
            ⚡ FastText Verify
        </div>
        <div style="color:#6b6b8a">→</div>
        <div style="background:#43e97b22; border:1px solid #43e97b44; border-radius:8px; padding:6px 14px; font-size:0.85rem">
            🌍 Helsinki NLP
        </div>
        <div style="color:#6b6b8a">→</div>
        <div style="background:#43e97b22; border:1px solid #43e97b44; border-radius:8px; padding:6px 14px; font-size:0.85rem">
            ✅ Output
        </div>
    </div>
</div>
""", unsafe_allow_html=True)

# ─────────────────────────────────────────────
# FOOTER
# ─────────────────────────────────────────────
st.markdown("""
<br>
<div style="text-align:center; padding: 1rem 0; color: #6b6b8a; font-size: 0.8rem;">
    Built with ❤️ using Scikit-Learn · FastText · Helsinki-NLP · Streamlit
</div>
""", unsafe_allow_html=True)

ModuleNotFoundError: No module named 'streamlit'